**This notebook transforms the Bronze sales data into cleansed, analytics ready silver layer. At this stage data quality and consistency are enforced by examining missing values, converting Order_Date into proper timestampt and enriching the dataset with month and year dimensions. Duplicate records are identified using Order_ID and Product as business key and by retaining the most recent record per key. The resulting Silver dataset is written in Parquet format and partitioned by year and month in order to maximise the efficiency and maintain data integrity.**

In [1]:
# Import the necessary libraries 
from pyspark.sql import SparkSession
from pathlib import Path
import src.sqlqueries as sq
import sys
import os
import warnings
import utils.logger as logger
from pyspark.sql import functions as F
from pyspark.sql.functions import to_timestamp, col
from pyspark.sql.functions import sum as spark_sum, when
warnings.filterwarnings("ignore")


#Set the path for logging outputs
job_name = "sales_ETL"
data_base_path = Path("../Logs") # path for logging data
data_working_path = os.path.join(data_base_path, job_name) 
os.makedirs(data_working_path, exist_ok=True)
logger.set_logging_path(data_working_path)


# Spark initialization locally for development
spark = (
    SparkSession.builder
    .appName("sales-bronze-ingestion")
    .getOrCreate()
)

logger.log("Spark Session initialized")

df = spark.read.parquet(
    "../data/bronze/sales"
)
logger.log("Spark DataFrame created from bronze sales parquet files")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/03 02:44:30 WARN Utils: Your hostname, gvidias, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/02/03 02:44:30 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/03 02:44:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/02/03 02:44:40 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


2026-02-03 02:44:42: Spark Session initialized
2026-02-03 02:44:43: Spark DataFrame created from bronze sales parquet files


# Heading Data Cleaning
**1. Examine Nulls. Note that data quality validations were implemented using conditional logging, ensuring that  meaningful anomalies (such as null patterns) are recorded.
2. Order_Date was initially set as a string - this column will be used for the partition strategy so it will cast to timestampt
3. Deal with duplicate values --> Duplicates were identified using the composite key (Order_ID, Product), assuming each product appears at most once per order.**

In [ ]:
df.createOrReplaceTempView("sales_data")
logger.log("Temporary view 'sales_data' created from sales df")

# Check for nulls and collect results
logger.log("Check for nulls in all rows of sales_data")
df_total_nulls  = spark.sql(sq.total_fully_null_rows)

logger.log("Check for nulls in each column of sales_data")
df_column_nulls  = spark.sql(sq.nulls_per_column)

total_null_rows = df_total_nulls.collect()[0][0]
column_nulls = df_column_nulls.collect()[0].asDict()

# Now log only is something is found
if total_null_rows > 0:
    logger.log(f"DATA QUALITY ISSUE | Fully null rows detected: {total_null_rows}")
else :
    logger.log("No fully null rows detected")

columns_with_nulls = {k: v for k, v in column_nulls.items() if v > 0}

if columns_with_nulls:
    logger.log(f"DATA QUALITY ISSUE | Nulls detected per column: {columns_with_nulls}")
else: 
    logger.log("No nulls detected in any column")

# Raise error if fully null rows exceed 50% of the dataset 
if total_null_rows > df.count() * 0.5:
    raise ValueError("Critical data quality issue: fully null rows detected")

#### A threshold based validation was implemented to fail the pipeline only when fully null rows exceed 50% of the dataset, preventing both false positives and silent catastrophic failures ####

# examine basic statistics
df.describe().show()
logger.log("Data profiling completed")


2026-02-03 02:44:44: Temporary view 'sales_data' created from sales df
2026-02-03 02:44:44: Check for nulls in all rows of sales_data
2026-02-03 02:44:44: Check for nulls in each column of sales_data
2026-02-03 02:44:46: No fully null rows detected
2026-02-03 02:44:46: No nulls detected in any column


26/02/03 02:44:46 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+-----------------+------------+-------------------+------------------+--------------+--------------------+
|summary|         Order_ID|     Product|   Quantity_Ordered|        Price_Each|    Order_Date|    Purchase_Address|
+-------+-----------------+------------+-------------------+------------------+--------------+--------------------+
|  count|           185950|      185950|             185950|            185950|        185950|              185950|
|   mean|230417.5693788653|        NULL| 1.1243828986286637|184.39973476743927|          NULL|                NULL|
| stddev|51512.73710999594|        NULL|0.44279262402866965|332.73132988434367|          NULL|                NULL|
|    min|           141234|20in Monitor|                  1|              2.99|01/01/19 03:07|1 11th St, Atlant...|
|    max|           319670|      iPhone|                  9|            1700.0|12/31/19 23:53|999 Wilson St, Sa...|
+-------+-----------------+------------+-------------------+------------

**The validation showed that the Order_Date column contains no null values. This is a critical result, as Order_Date will be used as the basis for time-based partitioning in downstream layers. Ensuring completeness at this stage guarantees that no records will be excluded or misrouted during partitioning. Given this validation, the column can be safely converted from string to timestamp format in preparation for Silver-layer transformations.**

In [3]:
df = df.withColumn(
    "Order_Date",
    to_timestamp(col("Order_Date"), "MM/dd/yy HH:mm")
)

# Create a data quality check when the date is null to raise an error
invalid_dates = df.select(spark_sum(when(col("Order_Date").isNull(), 1).otherwise(0)).alias("invalid_order_dates")).collect()[0][0]

if invalid_dates  > 0:
    raise ValueError("Critical data quality issue: Order_Date column contains invalid dates")
else:
    logger.log("Order_Date successfully converted to timestamp")

# Order_Date is can later be used to derive partition columns such as year and month without introducing skew or data loss
df = df.withColumn("Order_Year", F.year("Order_Date")).withColumn("Order_Month", F.month("Order_Date"))
logger.log("Order_Year and Order_Month columns created from Order_Date")

# Create a new temporary view with the enriched data
df.createOrReplaceTempView("sales_data_enriched")


2026-02-03 02:44:48: Order_Date successfully converted to timestamp
2026-02-03 02:44:48: Order_Year and Order_Month columns created from Order_Date


Assumption --> One order can contain multiple products 
           --> The same product should not appear multiple times in the same order

Duplicates were identified using the composite key (Order_ID, Product), assuming each product appears at most once per order and considering that the most recent timestamp is the most reliable version. Thus, when duplicates were detected, the most recent record based on Order_Date was retained.


In [4]:
logger.log("Check for duplicate rows in sales_data")
total_duplicates = spark.sql(sq.douplicates_count).collect()[0][0]
logger.log(f"Total duplicate rows detected: {total_duplicates}")

df = spark.sql(sq.remove_duplicates)
logger.log("Duplicate rows removed from sales_data_enriched")
df.createOrReplaceTempView("sales_data_enriched")

2026-02-03 02:44:48: Check for duplicate rows in sales_data


2026-02-03 02:44:50: Total duplicate rows detected: 264
2026-02-03 02:44:50: Duplicate rows removed from sales_data_enriched


Daily Partitioning was avoided to prevent thousands of folders over time, very small Parquet files per partition and worse performance for monthly queries which are very common. Year only partitioning isn't recommended also since queries for a single month still scan the full year thus, the performance will be bad. The best decision here is partitioning by Order_Year and Order_Month which matches common analytical access patterns (monthly trends, MTD, YTD), keeps partition count reasonable, produces sufficiently large Parquet files per partition and enables efficient pruning without small file explosion.

In [5]:
# Write Parquet file to silver - cleansed  layer 
logger.log("Start ingestion of Parquet file to cleansed layer")
(
    df.write
    .mode("overwrite")
    .partitionBy("Order_Year", "Order_Month")
    .parquet("../data/cleansed/sales")
)

logger.log("Cleansed sales data written to ../data/cleansed/sales partitioned by year and month")

2026-02-03 02:44:50: Start ingestion of Parquet file to cleansed layer


26/02/03 02:44:51 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/02/03 02:44:51 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/02/03 02:44:51 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/02/03 02:44:51 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/02/03 02:44:51 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/02/03 02:44:51 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/02/03 02:44:51 WARN MemoryManager: Total allocation exceeds 95.00% 

2026-02-03 02:44:52: Cleansed sales data written to ../data/cleansed/sales partitioned by year and month
